# 14 — RQ3 faithfulness and sanity

**Objective.** Run conditional deletion against random/bottom rankings, approximation controls, model-parameter and label randomization, semi-synthetic known-signal recovery, and isolated leakage detection.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Substantive “driver” language is blocked unless all prespecified positive and negative controls pass.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("14", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import json, joblib, yaml
import numpy as np
import pandas as pd
from cruxvc.calibration import CalibratedModel, PlattCalibrator
from cruxvc.controls import known_signal_recovery, randomize_fitted_model_parameters
from cruxvc.explanations import GowerKNNDonorSampler, active_feature_groups, explain_dataset, load_feature_groups
from cruxvc.faithfulness import faithfulness_audit, sanity_control_summary
from cruxvc.inference import construct_driver_taxonomy
from cruxvc.io import read_table, write_json, write_table
from cruxvc.manifest import append_test_access_log
from cruxvc.models import fit_model, predict_positive
from cruxvc.synthetic import inject_future_round_count_positive_control, randomize_labels, semi_synthetic_outcomes

append_test_access_log(
    P,
    stage_id="14",
    purpose="locked explanation controls and faithfulness",
    resources=[P.attributions / "final_attributions_long.parquet"],
)
features = read_table(P.processed / "features_strict.parquet")
cohort = read_table(P.processed / "cohort_labels.parquet")
splits = read_table(P.protocol / "split_ids.parquet")
calibrated_registry = read_table(P.models / "calibration_registry.parquet")
final_attr = read_table(P.attributions / "final_attributions_long.parquet")
data = (
    features.merge(cohort, on=["case_id", "company_permalink", "t0"])
    .merge(splits[["case_id", "time_block"]], on="case_id")
)
development = data[data["time_block"].astype(str).eq("development")].copy()
cal2008 = data[data["time_block"].astype(str).eq("probability_calibration")].copy()
test = data[data["time_block"].astype(str).eq("final_test")].copy()
audit_ids = pd.read_csv(P.protocol / "local_audit_case_ids.csv")["case_id"]
cases = test[test["case_id"].isin(audit_ids)].copy()
feature_columns = [c for c in features.columns if c not in {"case_id", "company_permalink", "t0"}]
groups = active_feature_groups(load_feature_groups(P.config / "feature_groups.yaml"), feature_columns)
statistical = yaml.safe_load((P.config / "statistical_analysis.yaml").read_text(encoding="utf-8"))

In [ ]:
faithfulness_job = (
    calibrated_registry[
        calibrated_registry["analysis_role"].eq("matched_reference")
        & calibrated_registry["outcome"].eq("F36")
    ]
    .sort_values("platt_oof_log_loss")
    .iloc[0]
)
model = joblib.load(faithfulness_job["calibrated_model_path"])
model_attr = final_attr[
    final_attr["outcome"].eq("F36")
    & final_attr["family"].eq(faithfulness_job["family"])
    & final_attr["config_id"].eq(faithfulness_job["config_id"])
    & final_attr["refit_id"].eq("deployment")
    & final_attr["analysis_role"].eq("matched_reference_deployment")
].copy()
if model_attr.empty:
    raise RuntimeError("No matched-reference deployment attributions found for the faithfulness model")
sampler = GowerKNNDonorSampler.fit(
    development,
    feature_columns,
    k=int(statistical["faithfulness"]["conditional_sampler_primary_k"]),
)
curves, faithfulness = faithfulness_audit(
    model,
    cases,
    model_attr,
    groups,
    sampler,
    feature_columns,
    steps=tuple(statistical["faithfulness"]["deletion_steps"]),
    random_repetitions=int(statistical["faithfulness"]["random_rank_repetitions"]),
    seed=int(CFG["execution"]["random_seed"]) + 140,
)
curves_path = write_table(curves, P.controls / "rq3_conditional_deletion_curves.parquet")
faithfulness_path = write_table(
    faithfulness,
    P.controls / "rq3_faithfulness_case_summary.parquet",
)

In [ ]:
# Parameter- and label-randomization controls use a supported confirmatory
# family on development copies only. This choice is separate from the model used
# for the headline faithfulness audit and is recorded explicitly.
supported_families = ["logistic_elastic_net", "random_forest", "explainable_boosting"]
control_job = (
    calibrated_registry[
        calibrated_registry["analysis_role"].eq("matched_reference")
        & calibrated_registry["outcome"].eq("F36")
        & calibrated_registry["family"].isin(supported_families)
    ]
    .sort_values(["platt_oof_log_loss", "family", "config_id"])
    .iloc[0]
)
control_model = joblib.load(control_job["calibrated_model_path"])
control_model_attr = final_attr[
    final_attr["outcome"].eq("F36")
    & final_attr["family"].eq(control_job["family"])
    & final_attr["config_id"].eq(control_job["config_id"])
    & final_attr["refit_id"].eq("deployment")
    & final_attr["analysis_role"].eq("matched_reference_deployment")
].copy()
control_cases = cases.head(min(60, len(cases)))
background_ids = pd.read_csv(P.protocol / "explanation_background_ids.csv")
first_background_id = background_ids["background_id"].iloc[0]
first_bg = (
    background_ids[background_ids["background_id"].eq(first_background_id)]
    .sort_values("order")["case_id"]
)
background = development[development["case_id"].isin(first_bg)].copy()
parameter_random_base = randomize_fitted_model_parameters(
    control_model.base_model,
    seed=int(CFG["execution"]["random_seed"]) + 141,
)
parameter_random_model = CalibratedModel(
    parameter_random_base,
    control_model.calibrator,
    tuple(feature_columns),
)
parameter_attr = explain_dataset(
    parameter_random_model,
    control_cases,
    background,
    groups,
    feature_columns,
    model_metadata={
        "outcome": "F36",
        "family": control_job["family"],
        "config_id": control_job["config_id"],
        "model_id": "parameter_randomized",
        "refit_id": "control",
    },
    background_id="control_bg",
    approximation_seed=141,
    n_orderings=max(8, int(PROFILE["permutation_orderings"])),
)
params = json.loads(control_job["parameters_json"])
randomized_y = randomize_labels(
    development["F36"],
    seed=int(CFG["execution"]["random_seed"]) + 142,
)
label_base = fit_model(
    development,
    randomized_y,
    feature_columns,
    control_job["family"],
    params,
    int(control_job["seed"]),
)
label_raw = predict_positive(label_base, cal2008, feature_columns)
label_calibrator = PlattCalibrator().fit(
    label_raw,
    randomize_labels(cal2008["F36"], seed=int(CFG["execution"]["random_seed"]) + 143),
)
label_model = CalibratedModel(label_base, label_calibrator, tuple(feature_columns))
label_attr = explain_dataset(
    label_model,
    control_cases,
    background,
    groups,
    feature_columns,
    model_metadata={
        "outcome": "F36",
        "family": control_job["family"],
        "config_id": control_job["config_id"],
        "model_id": "label_randomized",
        "refit_id": "control",
    },
    background_id="control_bg",
    approximation_seed=142,
    n_orderings=max(8, int(PROFILE["permutation_orderings"])),
)
real_control_attr = control_model_attr[
    control_model_attr["case_id"].isin(control_cases["case_id"])
].copy()
parameter_summary = sanity_control_summary(real_control_attr, parameter_attr).assign(
    control="parameter_randomization"
)
label_summary = sanity_control_summary(real_control_attr, label_attr).assign(
    control="label_randomization"
)
randomization_summary = pd.concat([parameter_summary, label_summary], ignore_index=True)

In [ ]:
# Semi-synthetic known-signal and isolated future-information positive controls.
synthetic_labels, truth = semi_synthetic_outcomes(
    development, groups,
    shared_groups=["initial_financing_amount", "investor_experience"],
    outcome_specific_groups={"SYN_A": ["calendar_regime"], "SYN_B": ["investor_network_position"]},
    seed=int(CFG["execution"]["random_seed"]) + 144,
)
synthetic_data = development.merge(synthetic_labels, on="case_id")
synthetic_parts = []
for synthetic_outcome in ["SYN_A", "SYN_B"]:
    synthetic_model = fit_model(synthetic_data, synthetic_data[synthetic_outcome], feature_columns, "logistic_elastic_net", {"C": 1.0, "l1_ratio": 0.5}, 144)
    synthetic_parts.append(explain_dataset(
        synthetic_model, synthetic_data.head(min(100, len(synthetic_data))), background, groups, feature_columns,
        model_metadata={"outcome": synthetic_outcome, "family": "logistic_elastic_net", "config_id": "semi_synthetic", "model_id": synthetic_outcome, "refit_id": "control"},
        background_id="control_bg", approximation_seed=144, n_orderings=max(8, int(PROFILE["permutation_orderings"])),
    ))
synthetic_attr = pd.concat(synthetic_parts, ignore_index=True)
recovery = known_signal_recovery(synthetic_attr, truth)

leakage_matrix = inject_future_round_count_positive_control(development[features.columns], cohort[["case_id", "F36"]], source_outcome="F36", seed=145)
leakage_features = feature_columns + ["future_round_count_36"]
leakage_groups = dict(groups)
leakage_groups["injected_future_leakage"] = ["future_round_count_36"]
leakage_model = fit_model(leakage_matrix, development["F36"], leakage_features, "random_forest", {"n_estimators": 500, "max_depth": 6, "min_samples_leaf": 5, "max_features": "sqrt"}, 145)
leakage_background = leakage_matrix.head(min(100, len(leakage_matrix)))
leakage_attr = explain_dataset(
    leakage_model, leakage_matrix.head(min(100, len(leakage_matrix))), leakage_background, leakage_groups, leakage_features,
    model_metadata={"outcome": "F36", "family": "random_forest", "config_id": "isolated_leakage_control", "model_id": "isolated_leakage_control", "refit_id": "control"},
    background_id="control_bg", approximation_seed=145, n_orderings=max(8, int(PROFILE["permutation_orderings"])),
)
leakage_rank = leakage_attr.groupby("feature_group")["absolute_phi"].mean().sort_values(ascending=False).reset_index(name="mean_absolute_phi")

In [ ]:
randomization_path = write_table(
    randomization_summary,
    P.controls / "rq3_randomization_controls.csv",
)
recovery_path = write_table(recovery, P.controls / "rq3_semisynthetic_recovery.csv")
leakage_path = write_table(
    leakage_rank,
    P.controls / "rq3_injected_leakage_control.csv",
)
approximation = pd.read_csv(P.audits / "09_approximation_repeat_diagnostics.csv")
approximation_sd = float(approximation["approximation_sd"].fillna(0).mean())
minimum_faithfulness = float(
    statistical["faithfulness"]["minimum_top_vs_random_normalized_aopc"]
)
control_summary = pd.DataFrame(
    [
        {
            "control": "conditional_deletion_top_vs_random",
            "passes_control": bool(
                faithfulness["normalized_top_minus_random"].mean() >= minimum_faithfulness
            ),
            "value": float(faithfulness["normalized_top_minus_random"].mean()),
        },
        {
            "control": "conditional_deletion_top_vs_bottom",
            "passes_control": bool(faithfulness["top_minus_bottom_aopc"].mean() > 0),
            "value": float(faithfulness["top_minus_bottom_aopc"].mean()),
        },
        {
            "control": "parameter_randomization",
            "passes_control": bool(parameter_summary["passes_control"].all()),
            "value": float(parameter_summary["mean_absolute_change"].mean()),
        },
        {
            "control": "label_randomization",
            "passes_control": bool(label_summary["passes_control"].all()),
            "value": float(label_summary["mean_absolute_change"].mean()),
        },
        {
            "control": "semi_synthetic_signal_recovery",
            "passes_control": bool(recovery["top_k_inclusion"].mean() > 0.50),
            "value": float(recovery["top_k_inclusion"].mean()),
        },
        {
            "control": "injected_leakage_detection",
            "passes_control": bool(
                not leakage_rank.empty
                and leakage_rank.iloc[0]["feature_group"] == "injected_future_leakage"
            ),
            "value": float(leakage_rank.iloc[0]["mean_absolute_phi"]),
        },
        {
            "control": "approximation_repeat_precision",
            "passes_control": bool(
                approximation_sd
                < float(statistical["smallest_meaningful_effects"]["rq1_delta_spec"])
            ),
            "value": approximation_sd,
        },
    ]
)
summary_path = write_table(control_summary, P.controls / "rq3_control_summary.csv")
# Final driver states are released only after all mandatory controls are available.
deployment_attr = final_attr[
    final_attr["analysis_role"].eq("label_specific_near_optimal_deployment")
].copy()
taxonomy = construct_driver_taxonomy(deployment_attr, control_summary)
taxonomy_path = write_table(
    taxonomy,
    P.inference / "construct_robust_driver_taxonomy.csv",
)
CTX.recorder.complete(
    [
        curves_path,
        faithfulness_path,
        randomization_path,
        recovery_path,
        leakage_path,
        summary_path,
        taxonomy_path,
    ],
    extra={
        "faithfulness_model_family": str(faithfulness_job["family"]),
        "randomization_control_family": str(control_job["family"]),
    },
)
print(control_summary.to_string(index=False))
print(taxonomy.to_string(index=False))